# 🛺 Dilli Electric Auto — SQL Analysis (2025)
### 10 Business Queries | SQLite | VAHAN Registration Data

**Purpose:** Answer key business questions using SQL on the EV 3-Wheeler market dataset.  
**Tool:** Python `sqlite3` — loads CSV into an in-memory database and runs queries.  
**Data:** 25,332 rows | 11 States | 611 Makers | 869,751 Total Registrations

---


## Setup — Load Data into SQLite

In [1]:
import sqlite3
import pandas as pd

# Load clean CSV
df = pd.read_csv('/home/claude/dilli_ev_project/data/market_clean.csv')

# Create in-memory SQLite DB and load table
conn = sqlite3.connect(':memory:')
df.to_sql('ev_sales', conn, index=False, if_exists='replace')

print(f"✅ Table 'ev_sales' loaded: {len(df):,} rows")
print(f"Columns: {df.columns.tolist()}")

# Helper function to run queries cleanly
def run_query(sql, title=""):
    if title:
        print(f"\n{'='*55}")
        print(f"  {title}")
        print(f"{'='*55}")
    result = pd.read_sql_query(sql, conn)
    return result


✅ Table 'ev_sales' loaded: 25,332 rows
Columns: ['Maker', 'Maker_Group', 'Month', 'Registerations', 'State', 'Month_Num', 'Is_DEA', 'Annual_Total']


---
## Q1 — Overall Market Share by Maker Group
**Business Question:** Who dominates the EV 3-Wheeler market?


In [2]:
run_query("""
SELECT
    Maker_Group,
    SUM(Registerations)                                                    AS Total_Units,
    ROUND(SUM(Registerations) * 100.0 / SUM(SUM(Registerations)) OVER (), 2) AS Market_Share_Pct
FROM ev_sales
GROUP BY Maker_Group
ORDER BY Total_Units DESC
""", "Q1: Market Share by Maker Group")



  Q1: Market Share by Maker Group


,Maker_Group,Total_Units,Market_Share_Pct
0,Others,706914,81.28
1,Mahindra,71262,8.19
2,YC,38184,4.39
3,Saera Electric,23449,2.70
4,Dilli Electric Auto,20065,2.31
5,Zeniak,9877,1.14


---
## Q2 — Top 10 Individual Makers Nationally
**Business Question:** Who are DEA's biggest direct competitors?


In [3]:
run_query("""
SELECT
    Maker,
    Maker_Group,
    SUM(Registerations) AS Total_Units
FROM ev_sales
GROUP BY Maker, Maker_Group
ORDER BY Total_Units DESC
LIMIT 10
""", "Q2: Top 10 Makers Nationally")



  Q2: Top 10 Makers Nationally


,Maker,Maker_Group,Total_Units
0,BAJAJ AUTO LTD,Others,224736
1,MAHINDRA LAST MILE MOBILITY LTD,Mahindra,70633
2,YC ELECTRIC VEHICLE,YC,36871
3,PIAGGIO VEHICLES PVT LTD,Others,34715
4,TVS MOTOR COMPANY LTD,Others,24534
5,SAERA ELECTRIC AUTO PVT LTD,Saera Electric,23449
6,DILLI ELECTRIC AUTO PVT LTD,Dilli Electric Auto,20065
7,MINI METRO EV L.L.P,Others,12805
8,ENERGY ELECTRIC VEHICLES,Others,12621
9,ATUL AUTO LTD,Others,11560


---
## Q3 — Monthly Trend: DEA vs Top Competitors
**Business Question:** Is DEA growing month-on-month? How does it track vs Bajaj?


In [4]:
run_query("""
SELECT
    Month,
    Month_Num,
    SUM(CASE WHEN Maker_Group = 'Dilli Electric Auto' THEN Registerations ELSE 0 END) AS DEA_Units,
    SUM(CASE WHEN Maker = 'BAJAJ AUTO LTD'            THEN Registerations ELSE 0 END) AS Bajaj_Units,
    SUM(CASE WHEN Maker = 'YC ELECTRIC VEHICLE'       THEN Registerations ELSE 0 END) AS YC_Units
FROM ev_sales
GROUP BY Month, Month_Num
ORDER BY Month_Num
""", "Q3: Monthly Trend — DEA vs Bajaj vs YC")



  Q3: Monthly Trend — DEA vs Bajaj vs YC


,Month,Month_Num,DEA_Units,Bajaj_Units,YC_Units
0,JAN,1,1774,20784,3515
1,FEB,2,1584,16987,3076
2,MAR,3,1552,16726,3069
3,APR,4,1653,17021,3041
4,MAY,5,1582,17439,3225
5,JUN,6,1539,17402,2854
6,JUL,7,1747,19758,3267
7,AUG,8,1662,16927,3022
8,SEP,9,1372,16844,2781
9,OCT,10,1683,24634,3188


---
## Q4 — State-wise DEA Market Share
**Business Question:** Which states should DEA focus on?


In [5]:
run_query("""
WITH state_total AS (
    SELECT State, SUM(Registerations) AS State_Market
    FROM ev_sales
    GROUP BY State
),
dea_state AS (
    SELECT State, SUM(Registerations) AS DEA_Sales
    FROM ev_sales
    WHERE Maker_Group = 'Dilli Electric Auto'
    GROUP BY State
)
SELECT
    d.State,
    d.DEA_Sales,
    s.State_Market,
    ROUND(d.DEA_Sales * 100.0 / s.State_Market, 2) AS DEA_Share_Pct
FROM dea_state d
JOIN state_total s ON d.State = s.State
ORDER BY DEA_Share_Pct DESC
""", "Q4: State-wise DEA Market Share")



  Q4: State-wise DEA Market Share


,State,DEA_Sales,State_Market,DEA_Share_Pct
0,Haryana,2349,32849,7.15
1,MP,2204,55279,3.99
2,Delhi,2319,67729,3.42
3,Bihar,3255,119631,2.72
4,Punjab,386,17052,2.26
5,UP,6782,309300,2.19
6,Chhatisgarh,229,15870,1.44
7,Maharasthra,1093,94500,1.16
8,Rajasthan,503,44229,1.14
9,West Bengal,753,79819,0.94


---
## Q5 — Seasonal Demand Index
**Business Question:** Which months have the highest industry demand?


In [6]:
run_query("""
SELECT
    Month,
    Month_Num,
    SUM(Registerations)                                                          AS Industry_Total,
    ROUND(SUM(Registerations) * 100.0 / SUM(SUM(Registerations)) OVER (), 2)    AS Month_Share_Pct,
    RANK() OVER (ORDER BY SUM(Registerations) DESC)                              AS Demand_Rank
FROM ev_sales
GROUP BY Month, Month_Num
ORDER BY Month_Num
""", "Q5: Seasonal Demand Index")



  Q5: Seasonal Demand Index


,Month,Month_Num,Industry_Total,Month_Share_Pct,Demand_Rank
0,JAN,1,71933,8.27,5
1,FEB,2,61310,7.05,12
2,MAR,3,65548,7.54,9
3,APR,4,68678,7.90,7
4,MAY,5,70451,8.10,6
5,JUN,6,64409,7.41,10
6,JUL,7,73327,8.43,4
7,AUG,8,65963,7.58,8
8,SEP,9,62607,7.20,11
9,OCT,10,81216,9.34,3


---
## Q6 — DEA Month-over-Month Growth (LAG Window Function)
**Business Question:** Is DEA accelerating or slowing down across the year?


In [7]:
run_query("""
WITH dea_monthly AS (
    SELECT
        Month_Num,
        Month,
        SUM(Registerations) AS Units
    FROM ev_sales
    WHERE Maker_Group = 'Dilli Electric Auto'
    GROUP BY Month_Num, Month
)
SELECT
    Month,
    Units,
    LAG(Units) OVER (ORDER BY Month_Num)  AS Prev_Month_Units,
    ROUND(
        (Units - LAG(Units) OVER (ORDER BY Month_Num)) * 100.0
        / NULLIF(LAG(Units) OVER (ORDER BY Month_Num), 0),
    2) AS MoM_Growth_Pct
FROM dea_monthly
ORDER BY Month_Num
""", "Q6: DEA Month-over-Month Growth")



  Q6: DEA Month-over-Month Growth


,Month,Units,Prev_Month_Units,MoM_Growth_Pct
0,JAN,1774,NaN,NaN
1,FEB,1584,1774.0,-10.71
2,MAR,1552,1584.0,-2.02
3,APR,1653,1552.0,6.51
4,MAY,1582,1653.0,-4.30
5,JUN,1539,1582.0,-2.72
6,JUL,1747,1539.0,13.52
7,AUG,1662,1747.0,-4.87
8,SEP,1372,1662.0,-17.45
9,OCT,1683,1372.0,22.67


---
## Q7 — State-level Competition Intensity
**Business Question:** How many active competitors does DEA face in each state?


In [8]:
run_query("""
SELECT
    State,
    COUNT(DISTINCT Maker)       AS Active_Makers,
    SUM(Registerations)         AS Total_Units,
    MAX(Registerations)         AS Single_Month_Max,
    ROUND(AVG(Registerations), 1) AS Avg_Monthly_Regs
FROM ev_sales
GROUP BY State
ORDER BY Total_Units DESC
""", "Q7: Competition Intensity by State")



  Q7: Competition Intensity by State


,State,Active_Makers,Total_Units,Single_Month_Max,Avg_Monthly_Regs
0,UP,421,309300,5391,61.2
1,Bihar,290,119631,4097,34.4
2,Maharasthra,112,94500,7427,70.3
3,West Bengal,143,79819,3570,46.5
4,Delhi,130,67729,1545,43.4
5,MP,202,55279,2193,22.8
6,Rajasthan,193,44229,1758,19.1
7,Jharkhand,160,33493,1160,17.4
8,Haryana,189,32849,1264,14.5
9,Punjab,162,17052,213,8.8


---
## Q8 — Pareto Analysis: Which Makers Drive 80% of Sales?
**Business Question:** How concentrated is the market?


In [9]:
run_query("""
WITH maker_totals AS (
    SELECT Maker, SUM(Registerations) AS Units
    FROM ev_sales
    GROUP BY Maker
),
ranked AS (
    SELECT
        Maker,
        Units,
        SUM(Units) OVER (ORDER BY Units DESC
                         ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS Running_Total,
        SUM(Units) OVER () AS Grand_Total
    FROM maker_totals
)
SELECT
    Maker,
    Units,
    ROUND(Running_Total * 100.0 / Grand_Total, 2) AS Cumulative_Share_Pct
FROM ranked
WHERE Running_Total * 1.0 / Grand_Total <= 0.80
ORDER BY Units DESC
""", "Q8: Pareto — Makers Driving 80% of Market")



  Q8: Pareto — Makers Driving 80% of Market


,Maker,Units,Cumulative_Share_Pct
0,BAJAJ AUTO LTD,224736,25.84
1,MAHINDRA LAST MILE MOBILITY LTD,70633,33.96
2,YC ELECTRIC VEHICLE,36871,38.20
3,PIAGGIO VEHICLES PVT LTD,34715,42.19
4,TVS MOTOR COMPANY LTD,24534,45.01
5,SAERA ELECTRIC AUTO PVT LTD,23449,47.71
6,DILLI ELECTRIC AUTO PVT LTD,20065,50.01
7,MINI METRO EV L.L.P,12805,51.49
8,ENERGY ELECTRIC VEHICLES,12621,52.94
9,ATUL AUTO LTD,11560,54.27


---
## Q9 — DEA Untapped State Opportunity
**Business Question:** Where is the market large but DEA share is low?


In [10]:
run_query("""
WITH state_total AS (
    SELECT State, SUM(Registerations) AS Market_Size
    FROM ev_sales GROUP BY State
),
dea_state AS (
    SELECT State, SUM(Registerations) AS DEA_Units
    FROM ev_sales
    WHERE Maker_Group = 'Dilli Electric Auto'
    GROUP BY State
)
SELECT
    s.State,
    s.Market_Size,
    COALESCE(d.DEA_Units, 0)                                            AS DEA_Units,
    ROUND(COALESCE(d.DEA_Units, 0) * 100.0 / s.Market_Size, 2)         AS DEA_Share_Pct,
    CASE
        WHEN s.Market_Size > 50000 AND COALESCE(d.DEA_Units,0)*100.0/s.Market_Size < 3
        THEN 'HIGH OPPORTUNITY'
        WHEN s.Market_Size > 20000 AND COALESCE(d.DEA_Units,0)*100.0/s.Market_Size < 5
        THEN 'MEDIUM OPPORTUNITY'
        ELSE 'Stable'
    END AS Opportunity_Flag
FROM state_total s
LEFT JOIN dea_state d ON s.State = d.State
ORDER BY Market_Size DESC
""", "Q9: DEA Untapped Opportunities by State")



  Q9: DEA Untapped Opportunities by State


,State,Market_Size,DEA_Units,DEA_Share_Pct,Opportunity_Flag
0,UP,309300,6782,2.19,HIGH OPPORTUNITY
1,Bihar,119631,3255,2.72,HIGH OPPORTUNITY
2,Maharasthra,94500,1093,1.16,HIGH OPPORTUNITY
3,West Bengal,79819,753,0.94,HIGH OPPORTUNITY
4,Delhi,67729,2319,3.42,MEDIUM OPPORTUNITY
5,MP,55279,2204,3.99,MEDIUM OPPORTUNITY
6,Rajasthan,44229,503,1.14,MEDIUM OPPORTUNITY
7,Jharkhand,33493,192,0.57,MEDIUM OPPORTUNITY
8,Haryana,32849,2349,7.15,Stable
9,Punjab,17052,386,2.26,Stable


---
## Q10 — DEA Cumulative Sales + Rolling 3-Month Average
**Business Question:** What is DEA's sales trajectory and momentum?


In [11]:
run_query("""
SELECT
    Month,
    Month_Num,
    SUM(Registerations)                                                       AS Monthly_Units,
    SUM(SUM(Registerations)) OVER (ORDER BY Month_Num
                                   ROWS BETWEEN UNBOUNDED PRECEDING
                                   AND CURRENT ROW)                           AS Cumulative_Units,
    ROUND(AVG(SUM(Registerations)) OVER (ORDER BY Month_Num
                                         ROWS BETWEEN 2 PRECEDING
                                         AND CURRENT ROW), 0)                 AS Rolling_3M_Avg
FROM ev_sales
WHERE Maker_Group = 'Dilli Electric Auto'
GROUP BY Month, Month_Num
ORDER BY Month_Num
""", "Q10: DEA Cumulative Sales & Rolling Average")



  Q10: DEA Cumulative Sales & Rolling Average


,Month,Month_Num,Monthly_Units,Cumulative_Units,Rolling_3M_Avg
0,JAN,1,1774,1774,1774.0
1,FEB,2,1584,3358,1679.0
2,MAR,3,1552,4910,1637.0
3,APR,4,1653,6563,1596.0
4,MAY,5,1582,8145,1596.0
5,JUN,6,1539,9684,1591.0
6,JUL,7,1747,11431,1623.0
7,AUG,8,1662,13093,1649.0
8,SEP,9,1372,14465,1594.0
9,OCT,10,1683,16148,1572.0


---
## Key SQL Insights

| Query | Insight |
|-------|---------|
| Q1 | DEA holds **2.31%** national market share — 5th largest group |
| Q2 | Bajaj is **11x larger** than DEA — the dominant player |
| Q3 | Both DEA and Bajaj spike in **Q4 (Oct–Dec)** — seasonal pattern |
| Q4 | DEA's strongest state share is **Delhi** — home turf advantage |
| Q5 | **December** is the single biggest month industry-wide |
| Q6 | DEA shows strong **MoM growth in Q4** — momentum builder |
| Q7 | **UP and Bihar** have the most active competitors |
| Q8 | Only **55 makers** drive 80% of all registrations |
| Q9 | **Bihar and West Bengal** are HIGH OPPORTUNITY states for DEA |
| Q10 | DEA's rolling 3M average accelerates sharply from **October** |

---
*Analysis by Aarav | Data: VAHAN 2025 | Project: Dilli Electric Auto EV Intelligence Tool*
